# as-strided-windowing composite — cx1: derive Conv2d output shape, then build the matching as_strided patch view

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `as-strided-windowing`, `conv-output-shape`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "as-strided-windowing"
DD_ATOM_IDS = ["as-strided-windowing", "conv-output-shape"]
DD_SUBTOPICS = ["PyTorch: as_strided windowing", "CNN: Conv output shape"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Inside `conv2d_minimal` two atoms are joined at the hip:

1. **Conv output shape** — the analytic formula
   `H_out = (H + 2*pad - K) // stride + 1` (same for W). This is just integer arithmetic — no tensors involved. It tells you HOW MANY patch origins fit on the (padded) image.
2. **as_strided windowing** — given those H_out / W_out counts, build a view of shape `(H_out, W_out, K, K)` over the (already-padded) image. The stride tuple uses `img.stride()` as the inner step and `img.stride() * stride` as the outer step.

The composition: compute `(H_out, W_out)` analytically, THEN use those numbers as the outer two dims of the `as_strided` size tuple. The view's outer shape MUST equal the conv output spatial dims — otherwise downstream einsum / channel-sum will silently produce the wrong number of outputs.

**Anatomy.**
- `H_out = (H + 2*pad - K) // stride + 1`
- `W_out = (W + 2*pad - K) // stride + 1`
- `img_p = F.pad(img, (pad, pad, pad, pad))`
- `sH, sW = img_p.stride()`  → for a contiguous padded image, `(Wp, 1)`
- `patches = img_p.as_strided((H_out, W_out, K, K), (sH*stride, sW*stride, sH, sW))`

### Composite Exercise — derive Conv2d output shape, then build the matching as_strided patch view

**Atoms exercised together**: `as-strided-windowing`, `conv-output-shape`

Implement `cx1_patches_from_shape(img, K, stride, pad)`.

- `img`: float tensor of shape `(H, W)` — a single-channel image, contiguous.
- `K`: kernel side length (int).
- `stride`: int stride along both spatial dims.
- `pad`: zero-padding width applied symmetrically on both spatial dims (use `F.pad(img, (pad, pad, pad, pad))`).

Return a tuple `((H_out, W_out), patches)`:
- `(H_out, W_out)`: int output spatial dims computed via the conv-output-shape formula.
- `patches`: float tensor view of shape `(H_out, W_out, K, K)` containing every `KxK` stride-`stride` window of the PADDED image. Must be a view (storage-shared with `F.pad(img, ...)` — store it before strided-viewing it).

1. **conv-output-shape atom** — compute `H_out` and `W_out` from `H, W, K, stride, pad`.
2. **Pad the image** — `img_p = F.pad(img, (pad, pad, pad, pad))`.
3. **as-strided-windowing atom** — call `img_p.as_strided(size, stride_tuple)` ONCE.

The test fuzzes random `(H, W, K, stride, pad)` and cross-checks against a nested-loop reference, and verifies the analytic shape matches the view's shape.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx1_patches_from_shape(img, K, stride, pad):
    raise NotImplementedError

def _test_cx1():
    # Case A: stride=1, pad=0 — classic im2col.
    img = t.arange(20, dtype=t.float32).reshape(4, 5)
    (H_out, W_out), patches = cx1_patches_from_shape(img, K=2, stride=1, pad=0)
    assert (H_out, W_out) == (3, 4), f'expected (3,4), got {(H_out, W_out)}'
    assert tuple(patches.shape) == (3, 4, 2, 2)
    assert t.allclose(patches[0, 0], t.tensor([[0.0, 1.0], [5.0, 6.0]]))
    assert t.allclose(patches[2, 3], t.tensor([[13.0, 14.0], [18.0, 19.0]]))

    # Case B: stride=2, pad=0 — every other origin.
    img = t.arange(36, dtype=t.float32).reshape(6, 6)
    (H_out, W_out), patches = cx1_patches_from_shape(img, K=3, stride=2, pad=0)
    # (6 + 0 - 3) // 2 + 1 = 2.
    assert (H_out, W_out) == (2, 2), f'expected (2,2), got {(H_out, W_out)}'
    assert tuple(patches.shape) == (2, 2, 3, 3)
    # Patch [0,0] starts at img[0,0]; patch [0,1] starts at img[0,2] (stride=2).
    assert t.allclose(patches[0, 0], img[0:3, 0:3])
    assert t.allclose(patches[0, 1], img[0:3, 2:5])
    assert t.allclose(patches[1, 0], img[2:5, 0:3])

    # Case C: stride=1, pad=1 — output spatial dims must equal H (same-conv style).
    img = t.arange(16, dtype=t.float32).reshape(4, 4)
    (H_out, W_out), patches = cx1_patches_from_shape(img, K=3, stride=1, pad=1)
    # (4 + 2 - 3) // 1 + 1 = 4.
    assert (H_out, W_out) == (4, 4), f'expected (4,4) for same-conv style, got {(H_out, W_out)}'
    assert tuple(patches.shape) == (4, 4, 3, 3)
    # Top-left patch is the padded corner: zeros in the first row/col.
    tl = patches[0, 0]
    assert tl[0, 0].item() == 0.0 and tl[0, 1].item() == 0.0 and tl[0, 2].item() == 0.0
    assert tl[1, 0].item() == 0.0 and tl[2, 0].item() == 0.0
    # Center of patch [0,0] reads img[0, 0].
    assert tl[1, 1].item() == img[0, 0].item()

    # Case D: fuzz vs nested-loop reference.
    rng = t.Generator().manual_seed(101)
    for (H, W, K, S, P) in [(7, 9, 3, 1, 0), (8, 8, 3, 2, 1), (10, 6, 2, 2, 0), (5, 5, 5, 1, 2)]:
        im = t.randn(H, W, generator=rng)
        (Ho, Wo), patches = cx1_patches_from_shape(im, K=K, stride=S, pad=P)
        # Analytic shape check.
        assert Ho == (H + 2 * P - K) // S + 1
        assert Wo == (W + 2 * P - K) // S + 1
        assert tuple(patches.shape) == (Ho, Wo, K, K)
        # Nested-loop reference on the padded image.
        import torch.nn.functional as F
        im_p = F.pad(im, (P, P, P, P))
        for i in range(Ho):
            for j in range(Wo):
                ref = im_p[i*S : i*S + K, j*S : j*S + K]
                assert t.allclose(patches[i, j], ref), f'patch[{i},{j}] mismatch'
    _dd_passed.add('cx1')

_test_cx1()

<details><summary>Show solution — cx1</summary>

```python
def cx1_patches_from_shape(img, K, stride, pad):
    H, W = img.shape
    # Atom A (conv-output-shape): the closed-form output spatial dims.
    H_out = (H + 2 * pad - K) // stride + 1
    W_out = (W + 2 * pad - K) // stride + 1
    # Pad first — the windowing view will read from the PADDED storage.
    img_p = F.pad(img, (pad, pad, pad, pad))
    sH, sW = img_p.stride()  # (W + 2*pad, 1) for a contiguous padded image
    # Atom B (as-strided-windowing): outer dims walk patch origins with `stride`;
    # inner dims walk inside one patch with the contiguous-image strides.
    patches = img_p.as_strided(
        size=(H_out, W_out, K, K),
        stride=(sH * stride, sW * stride, sH, sW),
    )
    return (H_out, W_out), patches
```

The two atoms have to AGREE on `(H_out, W_out)`. If your analytic formula is off-by-one and your stride tuple isn't, `as_strided` will silently read past the storage and produce garbage patches with no exception. Always compute the shape first, then size the view to match — never the other way around.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx1',
        'subtopics': ["PyTorch: as_strided windowing", "CNN: Conv output shape"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()